<a href="https://colab.research.google.com/github/seu-usuario/seu-repo/blob/main/analise_streaming_GABARITO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Estudo de Caso 3: Engajamento em Plataforma de Streaming — GABARITO

**Contexto:** Você é Analista de Dados na StreamData. O produto quer entender o comportamento por plano (Básico, Padrão, Premium) para retenção.

**Arquivo necessário:** `streamdata_usuarios.csv` (300 usuários, colunas: `plano`, `horas_assistidas`, `score_satisfacao`, `cancelou_assinatura`, `dispositivo_principal`).

**Como usar no Colab:**
1. Faça upload deste `.ipynb` + do `.csv` (`Arquivos > Fazer upload` ou código de upload na Etapa 1).
2. Execute em ordem com `Shift + Enter`.

---

## 📥 Etapa 1: Leitura e Diagnóstico dos Dados

Importar `pandas` e `matplotlib.pyplot`, carregar o CSV em `df_stream`, exibir 5 primeiras linhas e `.info()`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# No Colab, se o CSV ainda não estiver carregado, descomente as 2 linhas abaixo:
# from google.colab import files
# uploaded = files.upload()  # selecione streamdata_usuarios.csv

df_stream = pd.read_csv("streamdata_usuarios.csv", encoding="utf-8")

print("5 primeiras linhas:")
display(df_stream.head())
print("\nTipos de dados e nulos:")
df_stream.info()
print(f"\nDimensões: {df_stream.shape[0]} linhas × {df_stream.shape[1]} colunas")

## 📊 Etapa 2: Média de Horas Assistidas por Plano (Barras Horizontal)

**Pergunta:** Qual plano consome mais horas por mês?

Agrupar por `plano`, média de `horas_assistidas`, gráfico `plt.barh` com rótulos, título e grade no eixo X.

In [ ]:
# Agrupar por plano e calcular a média
media_horas = df_stream.groupby("plano")["horas_assistidas"].mean().sort_values()
print(media_horas.round(1))

# Gráfico de barras horizontal
plt.figure(figsize=(9, 5))
cores = ["#8ecae6", "#219ebc", "#023e8a"]
barras = plt.barh(media_horas.index, media_horas.values, color=cores, edgecolor="black")
plt.xlabel("Média de horas assistidas / mês", fontsize=12)
plt.ylabel("Plano", fontsize=12)
plt.title("Média de Horas Assistidas por Plano", fontsize=14, fontweight="bold")
plt.grid(axis="x", linestyle="--", alpha=0.6)
for i, v in enumerate(media_horas.values):
    plt.text(v + 0.5, i, f"{v:.1f}h", va="center", fontsize=10)
plt.tight_layout()
plt.show()

# 💡 Resposta esperada: Premium > Padrão > Básico (efeito simulado no CSV).

## 📊 Etapa 3: Relação entre Satisfação e Cancelamento (Histogramas Sobrepostos)

**Pergunta:** Usuários com menor `score_satisfacao` cancelam mais?

Filtrar em `cancelou == 'Sim'` e `cancelou == 'Não'`, dois `plt.hist` sobrepostos com `alpha=0.6` + legenda.

In [ ]:
# Filtrar os dois grupos
cancelaram = df_stream[df_stream["cancelou_assinatura"] == "Sim"]["score_satisfacao"]
ativos = df_stream[df_stream["cancelou_assinatura"] == "Não"]["score_satisfacao"]
print(f"Cancelaram: {len(cancelaram)} | Ativos: {len(ativos)}")
print(f"Satisfação média — cancelaram: {cancelaram.mean():.2f} | ativos: {ativos.mean():.2f}")

# Histogramas sobrepostos
plt.figure(figsize=(10, 6))
bins = range(1, 12)  # notas 1 a 10
plt.hist(ativos, bins=bins, alpha=0.6, label="Ativos (Não cancelou)", color="green", edgecolor="black", align="left")
plt.hist(cancelaram, bins=bins, alpha=0.6, label="Cancelaram (Sim)", color="red", edgecolor="black", align="left")
plt.title("Distribuição do Score de Satisfação: Ativos × Cancelados", fontsize=14, fontweight="bold")
plt.xlabel("Score de Satisfação (1–10)", fontsize=12)
plt.ylabel("Quantidade de usuários", fontsize=12)
plt.xticks(range(1, 11))
plt.legend()
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# 💡 Resposta: cancelados concentram-se nas notas baixas (1–5); ativos nas notas altas (7–10).

## 📊 Etapa 4: Distribuição dos Dispositivos Principais (Pizza com Destaque)

**Pergunta:** Quais dispositivos priorizar no app?

Contar usuários por `dispositivo_principal`, pizza com `explode` no mais usado e `%1.1f%%`.

In [ ]:
# Contagem por dispositivo
disp_counts = df_stream["dispositivo_principal"].value_counts()
print(disp_counts)

# Destacar a maior fatia
explode = [0.08 if v == disp_counts.max() else 0 for v in disp_counts.values]

plt.figure(figsize=(8, 8))
plt.pie(disp_counts.values, labels=disp_counts.index, autopct="%1.1f%%",
        startangle=90, explode=explode, shadow=False,
        textprops={"fontsize": 11})
plt.title("Usuários por Dispositivo Principal", fontsize=14, fontweight="bold")
plt.axis("equal")
plt.tight_layout()
plt.show()

## 🎓 Conclusão executiva

In [ ]:
print("=" * 60)
print("RELATÓRIO STREAMDATA — Jun/2024")
print("=" * 60)
print(f"Plano com mais horas: {media_horas.idxmax()} ({media_horas.max():.1f}h/mês)")
print(f"Taxa de cancelamento geral: {(len(cancelaram)/len(df_stream)*100):.1f}%")
print(f"Dispositivo líder: {disp_counts.idxmax()} ({disp_counts.max()/len(df_stream)*100:.1f}%)")
print("Recomendação: priorizar Smart TV + Smartphone; criar alerta de retenção para scores ≤ 5.")
print("=" * 60)

### 🚀 Desafio extra

1. Taxa de cancelamento por plano: `df_stream.groupby("plano")["cancelou_assinatura"].apply(lambda s: (s=="Sim").mean()*100)`
2. Boxplot horas por plano com seaborn.
3. Salvar com `plt.savefig("grafico.png", dpi=300, bbox_inches="tight")`.